In [17]:
# Update your environment to Python 3.10 as described above before running this cell
import subprocess
import pkg_resources

# Set up Cohere API key
cohere_api_key = "Tf8kXsq2AihMtvYqQRoiNQPluxpCjIYskVD87rYx"

def install_if_needed(package, version):
    '''Function to ensure that the libraries used are consistent to avoid errors.'''
    try:
        pkg = pkg_resources.get_distribution(package)
    except pkg_resources.DistributionNotFound:
        subprocess.check_call(["pip", "install", f"{package}=={version}"])

# Installing required packages
install_if_needed("langchain", "0.2.2")
install_if_needed("langchain-openai", "0.1.8")
install_if_needed("langchain-community", "0.2.3")
install_if_needed("unstructured", "0.14.4")
install_if_needed("chromadb", "0.5.0")


In [20]:
# Set your Cohere API key
import os
cohere_api_key = "Tf8kXsq2AihMtvYqQRoiNQPluxpCjIYskVD87rYx"

# Import the required packages
import cohere
import langchain
from langchain import PromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.document_loaders import UnstructuredHTMLLoader
from chromadb.utils.embedding_functions import CohereEmbeddingFunction

# Initialize Cohere client
client = cohere.Client(api_key=cohere_api_key)

# Define the Cohere embedding function for LangChain
embedding_function = CohereEmbeddingFunction(api_key=cohere_api_key)

# Example for using the embedding function in LangChain with Chroma
# Load and split documents
# document_loader = UnstructuredHTMLLoader("path_to_your_file.html")  # Load HTML files or other sources
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# docs = text_splitter.split_documents(document_loader.load())

# Store embeddings in Chroma database
vector_store = Chroma(embedding_function=embedding_function)

# Use the embedding function or prompt as needed for your application


In [28]:
# Set your Cohere API key to a variable
import os
cohere_api_key = "Tf8kXsq2AihMtvYqQRoiNQPluxpCjIYskVD87rYx"

# Import the required packages
import cohere
import langchain
from langchain import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.document_loaders import UnstructuredHTMLLoader

# Initialize Cohere client
client = cohere.Client(api_key=cohere_api_key)

# Custom embedding function class for Cohere
class CohereDocumentEmbeddingFunction:
    def __init__(self, api_key):
        self.client = cohere.Client(api_key)

    def embed_documents(self, texts):
        # Generate embeddings for multiple documents
        response = self.client.embed(texts=texts)
        return response.embeddings

    def embed_query(self, query):
        # Generate embedding for a query
        response = self.client.embed(texts=[query])
        return response.embeddings[0]

# Initialize custom Cohere embedding function for LangChain
embedding_function = CohereDocumentEmbeddingFunction(api_key=cohere_api_key)

# Load the HTML as a LangChain document loader
loader = UnstructuredHTMLLoader(file_path="/content/sample_data/data/mg-zs-warning-messages.html")
car_docs = loader.load()

# Initialize RecursiveCharacterTextSplitter to make chunks of HTML text
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# Split HTML document into chunks
splits = text_splitter.split_documents(car_docs)

# Initialize Chroma vectorstore with documents as splits using Cohere embeddings
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_function)

# Setup vectorstore as retriever
retriever = vectorstore.as_retriever()

# Define a RAG prompt
prompt = PromptTemplate(
    input_variables=['question', 'context'],
    template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"
)

# Define a function for text generation using Cohere (as there's no ChatOpenAI equivalent for Cohere)
def generate_answer(prompt_text):
    response = client.generate(
        model='command-xlarge',  # Correct model ID
        prompt=prompt_text,
        max_tokens=50,
        temperature=0.5
    )
    return response.generations[0].text.strip()

# Combine retriever, prompt, and text generation into a query chain
def rag_query(query):
    # Use similarity_search instead of retrieve
    context_docs = vectorstore.similarity_search(query, k=3)  # Get top 3 documents
    context = " ".join([doc.page_content for doc in context_docs])  # Use top 3 documents as context
    prompt_text = prompt.format(question=query, context=context)
    return generate_answer(prompt_text)

# Initialize a query
query = "The Gasoline Particular Filter Full warning has appeared. What does this mean and what should I do about it?"

# Invoke the query
answer = rag_query(query)
print(answer)


The Gasoline Particular Filter Full warning means that there is an issue with the gasoline particular filter and you must consult an MG Authorized Repairer as soon as possible. 
Unfortunately, I cannot provide any further advice, nor can I assist with any
